# 02 — Station Flow & Imbalance
**Question:** Which stations chronically empty or overflow, and which require daily van redistribution?

Key finding: Waterloo +16% net outflow, Hop Exchange −15% inflow — 8-year structural patterns.

In [ ]:
from sqlalchemy import create_engine
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

PROJECT = 'm2-dataset-study'
engine = create_engine(f'bigquery://{PROJECT}/london_bicycles')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

## 2.1 — Top structurally imbalanced stations

In [ ]:
sql = """
SELECT
    station_name,
    total_departures,
    total_arrivals,
    net_outflow,
    ROUND(imbalance_pct, 1) AS imbalance_pct,
    flow_role
FROM `m2-dataset-study.london_bicycles.dim_station`
WHERE is_structural_imbalance
  AND NOT has_capacity_error
ORDER BY ABS(net_outflow) DESC
LIMIT 15
"""
df = pd.read_sql(sql, engine)

fig, ax = plt.subplots(figsize=(12, 6))
colours = df['flow_role'].map({'source': '#E74C3C', 'sink': '#2980B9', 'balanced': '#95A5A6'})
bars = ax.barh(df['station_name'], df['net_outflow'], color=colours)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_xlabel('Net Outflow (departures − arrivals, full dataset)', fontsize=11)
ax.set_title('Structurally Imbalanced Stations\nRed = chronic source | Blue = chronic sink', fontsize=13, fontweight='bold')

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#E74C3C', label='Source (bikes leave)'),
    Patch(color='#2980B9', label='Sink (bikes accumulate)'),
], fontsize=10)
plt.tight_layout()
plt.savefig('../outputs/02_station_imbalance.png', bbox_inches='tight')
plt.show()
print(df[['station_name','net_outflow','imbalance_pct','flow_role']].to_string(index=False))

## 2.2 — Hourly net flow for Waterloo vs Hop Exchange

In [ ]:
sql2 = """
SELECT
    station_name,
    hour_of_day,
    day_type,
    SUM(net_flow) AS net_flow
FROM `m2-dataset-study.london_bicycles.mart_station_demand`
WHERE station_name IN ('Waterloo Station 2, Waterloo', 'Hop Exchange, The Borough')
GROUP BY station_name, hour_of_day, day_type
ORDER BY station_name, day_type, hour_of_day
"""
df2 = pd.read_sql(sql2, engine)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
for ax, day_type, colour in zip(axes, ['Weekday', 'Weekend'], ['steelblue', 'darkorange']):
    sub = df2[df2['day_type'] == day_type]
    for station, grp in sub.groupby('station_name'):
        label = station.split(',')[0]
        ax.plot(grp['hour_of_day'], grp['net_flow'], marker='o', label=label, linewidth=2)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(f'{day_type} — Hourly Net Flow', fontsize=12, fontweight='bold')
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Net Flow (departures − arrivals)')
    ax.legend(fontsize=9)
    ax.set_xticks(range(0, 24, 2))

plt.suptitle('Waterloo (source) vs Hop Exchange (sink) — Complementary Flow', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/02_waterloo_hop_exchange_flow.png', bbox_inches='tight')
plt.show()

## 2.3 — Redistribution priority table

In [ ]:
sql3 = """
SELECT
    station_name,
    total_departures,
    total_arrivals,
    net_outflow,
    ROUND(imbalance_pct, 1) AS imbalance_pct,
    flow_role,
    docks_count
FROM `m2-dataset-study.london_bicycles.dim_station`
WHERE is_structural_imbalance
  AND NOT has_capacity_error
ORDER BY ABS(net_outflow) DESC
"""
priority = pd.read_sql(sql3, engine)
priority['redistribution_action'] = priority['flow_role'].map({
    'source': 'Restock daily (bikes leave)',
    'sink':   'Empty daily (bikes pile up)'
})
print(priority[['station_name','net_outflow','imbalance_pct','redistribution_action']]
      .to_string(index=False))